In [4]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import chromadb

# For embeddings
from sentence_transformers import SentenceTransformer

# For vector database
import chromadb

# For LLM
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# For progress tracking
from tqdm import tqdm

print(" All libraries imported successfully!")
print(f"Current directory: {os.getcwd()}")

 All libraries imported successfully!
Current directory: c:\Users\PC\Desktop\rag-complaint-chatbot-10academy\notebooks


In [5]:
print("=" * 50)
print("STEP 1: LOADING PRE-BUILT VECTOR STORE")
print("=" * 50)

# Path to the downloaded file (GO UP ONE LEVEL)
file_path = '../data/raw/complaint_embeddings.parquet'

# Check if file exists
if not os.path.exists(file_path):
    print(f"❌ File not found at: {file_path}")
    print("Checking alternative locations...")
    
    # Try other possible locations
    alternatives = [
        'data/raw/complaint_embeddings.parquet',
        '../data/raw/complaint_embeddings.parquet',
        '../../data/raw/complaint_embeddings.parquet',
        'complaint_embeddings.parquet'
    ]
    
    for alt in alternatives:
        if os.path.exists(alt):
            print(f"✅ Found at: {alt}")
            file_path = alt
            break
    else:
        print("❌ File not found anywhere!")
        print("Please make sure the file is in data/raw/")
else:
    # Load the embeddings
    print(f"Loading from: {file_path}")
    df_embeddings = pd.read_parquet(file_path)
    
    print(f"\n✅ Loaded {len(df_embeddings):,} chunks!")
    print(f"Columns: {df_embeddings.columns.tolist()}")

STEP 1: LOADING PRE-BUILT VECTOR STORE
Loading from: ../data/raw/complaint_embeddings.parquet

✅ Loaded 1,375,327 chunks!
Columns: ['id', 'document', 'embedding', 'metadata']


In [8]:
import pandas as pd

# Load the file
df_test = pd.read_parquet('../data/raw/complaint_embeddings.parquet')

# Show columns
print("Columns in your file:")
print(df_test.columns.tolist())

# Show first row to see structure
print("\nFirst row sample:")
print(df_test.iloc[0])

# Check if 'embedding' column exists
if 'embedding' in df_test.columns:
    print(f"\n✅ Embedding column found!")
    print(f"Embedding type: {type(df_test['embedding'].iloc[0])}")
    print(f"Embedding length: {len(df_test['embedding'].iloc[0])}")
else:
    print("\n❌ No 'embedding' column found!")
    print("Looking for alternative column names...")
    
    # Find any column that might contain embeddings
    for col in df_test.columns:
        try:
            sample = df_test[col].iloc[0]
            if isinstance(sample, (list, np.ndarray)) and len(sample) > 10:
                print(f"Found possible embedding column: {col}")
                print(f"  Sample: {sample[:5]}...")
        except:
            pass

Columns in your file:
['id', 'document', 'embedding', 'metadata']

First row sample:
id                                                  14069121_0
document     a card was opened under my name by a fraudster...
embedding    [-0.04277738183736801, 0.025624370202422142, -...
metadata     {'chunk_index': 0, 'company': 'CITIBANK, N.A.'...
Name: 0, dtype: object

✅ Embedding column found!
Embedding type: <class 'numpy.ndarray'>
Embedding length: 384


In [9]:
print("=" * 50)
print("STEP 2: CREATING FULL CHROMADB")
print("=" * 50)

import chromadb
from tqdm import tqdm
import numpy as np
import json

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="../vector_store/full_chroma_db")

# Create collection
collection = chroma_client.get_or_create_collection(
    name="complaints_full",
    metadata={"hnsw:space": "cosine"}
)

print(f"✅ Collection: complaints_full")
print(f"Current count: {collection.count():,}")

# Only add if empty
if collection.count() == 0:
    print(f"\n📤 Adding {len(df_embeddings):,} chunks to ChromaDB...")
    print("⏰ This will take ~5-10 minutes...")
    
    batch_size = 1000
    total = len(df_embeddings)
    
    for i in tqdm(range(0, total, batch_size), desc="Adding batches"):
        batch_end = min(i + batch_size, total)
        batch = df_embeddings.iloc[i:batch_end]
        
        # Prepare batch
        batch_ids = batch['id'].astype(str).tolist()
        batch_documents = batch['document'].astype(str).tolist()
        
        # Embeddings as list of lists
        batch_embeddings = batch['embedding'].tolist()
        
        # Prepare metadata - extract from the metadata column
        batch_metadata = []
        for _, row in batch.iterrows():
            meta = row['metadata']
            if isinstance(meta, dict):
                # Clean up metadata - convert all values to strings
                cleaned_meta = {}
                for key, value in meta.items():
                    if value is not None:
                        cleaned_meta[key] = str(value)
                batch_metadata.append(cleaned_meta)
            else:
                # If metadata is not a dict, create basic metadata
                batch_metadata.append({"id": str(row['id'])})
        
        # Add to collection
        collection.add(
            ids=batch_ids,
            documents=batch_documents,
            embeddings=batch_embeddings,
            metadatas=batch_metadata
        )
    
    print(f"\n✅ All embeddings added!")
    print(f"Total items: {collection.count():,}")
else:
    print(f"\n✅ Vector store already exists with {collection.count():,} chunks")

STEP 2: CREATING FULL CHROMADB
✅ Collection: complaints_full
Current count: 0

📤 Adding 1,375,327 chunks to ChromaDB...
⏰ This will take ~5-10 minutes...


Adding batches: 100%|██████████| 1376/1376 [1:39:37<00:00,  4.34s/it]



✅ All embeddings added!
Total items: 1,375,327


In [10]:
print("=" * 50)
print("STEP 3: LOADING EMBEDDING MODEL")
print("=" * 50)

from sentence_transformers import SentenceTransformer

# Load the same model used for the embeddings
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✅ Model loaded: all-MiniLM-L6-v2")
print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Test it
test_text = "This is a test"
test_embedding = embedding_model.encode(test_text)
print(f"   Test embedding shape: {test_embedding.shape}")

STEP 3: LOADING EMBEDDING MODEL


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded: all-MiniLM-L6-v2
   Embedding dimension: 384
   Test embedding shape: (384,)


In [11]:
print("=" * 50)
print("STEP 4: IMPLEMENTING RETRIEVER")
print("=" * 50)

def retrieve_chunks(query, n_results=5):
    """
    Retrieve relevant chunks from the full vector store
    
    Args:
        query (str): User's question
        n_results (int): Number of chunks to retrieve (k=5)
    
    Returns:
        dict: Retrieved chunks with metadata
    """
    # Encode the query
    query_embedding = embedding_model.encode(query).tolist()
    
    # Search the vector store
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    
    return {
        'documents': results['documents'][0],
        'metadatas': results['metadatas'][0],
        'distances': results['distances'][0]
    }

# Test the retriever
test_query = "Why are customers unhappy with credit cards?"
print(f"Testing retriever with: {test_query}")
print("-" * 50)

retrieved = retrieve_chunks(test_query, n_results=3)

print(f"\n✅ Retrieved {len(retrieved['documents'])} chunks")
for i, (doc, meta, dist) in enumerate(zip(
    retrieved['documents'], 
    retrieved['metadatas'], 
    retrieved['distances']
)):
    product = meta.get('product_category', meta.get('product', 'Unknown'))
    issue = meta.get('issue', '')
    print(f"\nResult {i+1} (Distance: {dist:.4f}):")
    print(f"  Product: {product}")
    if issue:
        print(f"  Issue: {issue}")
    print(f"  Preview: {doc[:120]}...")

STEP 4: IMPLEMENTING RETRIEVER
Testing retriever with: Why are customers unhappy with credit cards?
--------------------------------------------------

✅ Retrieved 3 chunks

Result 1 (Distance: 0.2864):
  Product: Credit Card
  Issue: Other features, terms, or problems
  Preview: card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the...

Result 2 (Distance: 0.2887):
  Product: Credit Card
  Issue: Other features, terms, or problems
  Preview: creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so...

Result 3 (Distance: 0.2895):
  Product: Credit Card
  Issue: Fees or interest
  Preview: credit card companies think of their card holders. the indifferent response to a long-term card holder again came to me ...


In [12]:
print("=" * 50)
print("STEP 5: CONTEXT FORMATTING & PROMPT ENGINEERING")
print("=" * 50)

def format_context(retrieved, max_chars=2000):
    """Format retrieved chunks into context"""
    context_parts = []
    total_chars = 0
    
    for i, (doc, meta) in enumerate(zip(
        retrieved['documents'], 
        retrieved['metadatas']
    )):
        product = meta.get('product_category', meta.get('product', 'Unknown'))
        issue = meta.get('issue', '')
        
        prefix = f"[Source {i+1}] Product: {product}"
        if issue:
            prefix += f", Issue: {issue}"
        prefix += "\n"
        
        chunk_text = prefix + doc + "\n\n"
        
        if total_chars + len(chunk_text) > max_chars:
            break
        
        context_parts.append(chunk_text)
        total_chars += len(chunk_text)
    
    return "\n".join(context_parts)

def create_prompt(question, context):
    """Create prompt following the assignment template"""
    return f"""You are a financial analyst assistant for CrediTrust Financial. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:
{context}

Question: {question}

Answer:"""

# Test
test_context = format_context(retrieved, max_chars=500)
test_prompt = create_prompt(test_query, test_context)
print("✅ Prompt template ready")
print("\nPrompt preview:")
print(test_prompt[:300] + "...")

STEP 5: CONTEXT FORMATTING & PROMPT ENGINEERING
✅ Prompt template ready

Prompt preview:
You are a financial analyst assistant for CrediTrust Financial. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:


Qu...


In [13]:
print("=" * 50)
print("STEP 6: LOADING PHI-2 LLM")
print("=" * 50)

model_name = "microsoft/phi-2"

print(f"Loading {model_name}...")
print("⏰ This may take 1-2 minutes...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype="auto"
)

generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(f"✅ Model loaded: {model_name}")

STEP 6: LOADING PHI-2 LLM
Loading microsoft/phi-2...
⏰ This may take 1-2 minutes...


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Model loaded: microsoft/phi-2


In [14]:
print("=" * 50)
print("STEP 7: BUILDING RAG PIPELINE")
print("=" * 50)

def rag_pipeline(question, n_results=5):
    """Complete RAG pipeline using the full dataset"""
    # Retrieve
    retrieved = retrieve_chunks(question, n_results)
    
    # Format
    context = format_context(retrieved, max_chars=2000)
    
    # Create prompt
    prompt = create_prompt(question, context)
    
    # Generate
    response = generator(prompt, max_new_tokens=300)[0]['generated_text']
    answer = response.split("Answer:")[-1].strip()
    
    return {
        'question': question,
        'answer': answer,
        'sources': retrieved['documents'],
        'source_metadata': retrieved['metadatas'],
        'context': context
    }

# Test
test_question = "Why are customers unhappy with credit cards?"
print(f"Testing: {test_question}")
print("-" * 50)

try:
    result = rag_pipeline(test_question)
    print(f"\n📝 Question: {result['question']}")
    print(f"\n💡 Answer: {result['answer'][:300]}...")
    print(f"\n📚 Sources: {len(result['sources'])} chunks")
except Exception as e:
    print(f"❌ Error: {e}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STEP 7: BUILDING RAG PIPELINE
Testing: Why are customers unhappy with credit cards?
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



📝 Question: Why are customers unhappy with credit cards?

💡 Answer: Customers are unhappy with credit cards because they feel that they are not given enough information about the features and benefits of the cards, they are not happy with the customer service provided by the card companies, they feel that the card companies are taking advantage of consumers who don'...

📚 Sources: 5 chunks


In [15]:
print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)

import pandas as pd
eval_df = pd.DataFrame(results)
print(eval_df.to_string(index=False))

# Save
eval_df.to_csv('../data/processed/full_dataset_evaluation.csv', index=False)
print("\n✅ Saved to: data/processed/full_dataset_evaluation.csv")

# Summary
print(f"\nTotal questions: {len(eval_df)}")
print(f"Average quality: {eval_df['Quality'].mean():.2f}/5")

EVALUATION RESULTS


NameError: name 'results' is not defined